# **IMBALANCE ANALYSIS AND MITIGATION**

## **IMPORTS**

In [1]:
# General imports
import pandas as pd
import numpy as np
from imblearn.pipeline import make_pipeline as imb_make_pipeline
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

# Preprocessor imports
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from preprocessing.preprocessor import preprocessor

# Oversampling imports
from imblearn.over_sampling import SMOTE, RandomOverSampler, ADASYN, BorderlineSMOTE

# Undersampling imports
from imblearn.under_sampling import RandomUnderSampler, EditedNearestNeighbours, NearMiss, TomekLinks

# Combination imports
from imblearn.combine import SMOTEENN, SMOTETomek

# Model imports
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, BaggingClassifier
import xgboost as xgb
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

## **DATASET LOADING**

In [2]:
# We read the partitions
X_train = pd.read_parquet("../data/cleaned/X_train.parquet")
X_test = pd.read_parquet("../data/cleaned/X_test.parquet")
y_train = pd.read_parquet("../data/cleaned/y_train.parquet").squeeze()
y_test = pd.read_parquet("../data/cleaned/y_test.parquet").squeeze()

# Check that all has been loaded correctly
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(10045, 32)
(10045,)
(4948, 32)
(4948,)


## **MODELS & TECHNIQUES**

### Models

In [3]:
# MODELS: default hyperparameters, random_state=42
MODELS = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Bagging': BaggingClassifier(random_state=42),
    'XGBoost': xgb.XGBClassifier(random_state=42, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1),
    'CatBoost': CatBoostClassifier(random_state=42, verbose=0, allow_writing_files=False)
}

### Balanced models

In [4]:
# MODELS_BALANCED: same models, class_weight='balanced' (where possible)
MODELS_BALANCED = {
    'Random Forest': RandomForestClassifier(random_state=42, class_weight='balanced'),
    'AdaBoost': AdaBoostClassifier(random_state=42),  # doesn't have class_weight
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),  # doesn't have class_weight 
    'Bagging': BaggingClassifier(random_state=42),  # doesn't have class_weight 
    'XGBoost': xgb.XGBClassifier(random_state=42, eval_metric='logloss'),  # doesn't have class_weight
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1, class_weight='balanced'),
    'CatBoost': CatBoostClassifier(random_state=42, verbose=0, auto_class_weights='Balanced', allow_writing_files=False)
}

### Balancing techniques

In [ ]:
# BALANCING_TECHNIQUES: techniques to mitigate the class imbalance
BALANCING_TECHNIQUES = {
    'None': 'passthrough',  # to compare without resampling
    'SMOTE (oversampling)': SMOTE(random_state=42),
    'RandomOverSampler (oversampling)': RandomOverSampler(random_state=42),
    'ADASYN (oversampling)': ADASYN(random_state=42, sampling_strategy='minority'),
    'BorderlineSMOTE (oversampling)': BorderlineSMOTE(random_state=42),
    'RandomUnderSampler (undersampling)': RandomUnderSampler(random_state=42),
    'EditedNearestNeighbours (undersampling)': EditedNearestNeighbours(),
    'NearMiss (undersampling)': NearMiss(),
    'TomekLinks (undersampling)': TomekLinks(),
    'SMOTEENN (combination)': SMOTEENN(random_state=42),
    'SMOTETomek (combination)': SMOTETomek(random_state=42)
}

### Initialization

In [6]:
# Initializing the folds that we will be using
skf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

# Initializing the results
results = []

## **IMBALANCE ANALYSIS**

In [7]:
target_distribution_train = y_train.value_counts()
target_distribution_test = y_test.value_counts()


for key, amount_train, amount_test in zip(target_distribution_train.index, target_distribution_train.values, target_distribution_test):
    print(f"""Class {key}: 
        Amount train: {amount_train} --> {round((amount_train/y_train.shape[0])*100, 2)}%
        Amount test: {amount_test} --> {round((amount_test/y_test.shape[0])*100, 2)}%""")


Class 4: 
        Amount train: 2812 --> 27.99%
        Amount test: 1385 --> 27.99%
Class 2: 
        Amount train: 2705 --> 26.93%
        Amount test: 1332 --> 26.92%
Class 3: 
        Amount train: 2183 --> 21.73%
        Amount test: 1076 --> 21.75%
Class 1: 
        Amount train: 2070 --> 20.61%
        Amount test: 1020 --> 20.61%
Class 0: 
        Amount train: 275 --> 2.74%
        Amount test: 135 --> 2.73%


## **IMBALANCE MITIGATION**

### Base models

In [42]:
# First we execute the base models

for model_name, model in MODELS_BALANCED.items():
    pipe = imb_make_pipeline(preprocessor, model)
    scores = cross_val_score(pipe, X_train, y_train, cv=skf, scoring="f1_macro") #f1_macro for imbalance data

    result = {"Model": f"{model_name} balanced", "balancing_technique": "no", "f1_mean": np.mean(scores), "f1_sd": np.std(scores)}
    results.append(result)
    
    print(f"Average f1-macro of {model_name}: {(round(np.mean(scores),2))} ")

Average f1-macro of Random Forest: 0.34 
Average f1-macro of AdaBoost: 0.28 
Average f1-macro of Gradient Boosting: 0.35 
Average f1-macro of Bagging: 0.32 
Average f1-macro of XGBoost: 0.34 


c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\iker\anaconda3\envs\machine_learning\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature nam

Average f1-macro of LightGBM: 0.34 
Average f1-macro of CatBoost: 0.35 


In [46]:
# Save partial results to a csv
partial_results = pd.DataFrame(results)

partial_results.to_parquet("technique_results/partial_results.parquet")


### Models & balancing techniques

In [32]:
# We check all the combinations of balancing techniques and models

for technique_name, technique in BALANCING_TECHNIQUES.items():
    for model_name, model in MODELS.items():
        pipe = imb_make_pipeline(preprocessor, technique, model)
        scores = cross_val_score(pipe, X_train, y_train, cv=skf, scoring="f1_macro") #f1_macro for imbalance data

        result = {"Model": f"{model_name}", "balancing_technique": technique_name, "f1_mean": np.mean(scores), "f1_sd": np.std(scores)}
        results.append(result)

        print(f"Average f1-macro of {model_name} using {technique_name}: {(round(np.mean(scores),2))} ") 



Average f1-macro of Random Forest using SMOTETomek (combination): 0.35 
Average f1-macro of AdaBoost using SMOTETomek (combination): 0.28 


KeyboardInterrupt: 

In [ ]:
# Save partial results to a csv
final_results = pd.DataFrame(results)
